<a href="https://colab.research.google.com/github/omairtahir3/flyrank_assignment01/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring, built on top of a binary classification signal.**

My lane (Refresh / Content Opportunity Scoring) ends in a *ranked queue*, a reviewer works down
an ordered list, not a single yes/no verdict. That output shape is scoring/ranking. But the score
itself is built from a classifier's probability output: a model that estimates, for each page,
"how likely is this page to carry a declining trend label." Sorting pages by that probability
(optionally blended with a transparent rule-based component, as the starter pipeline does) turns
a classification model into a ranking. So concretely: I train a **binary classifier**
(declining vs. not), and I **use its probability output as a ranking score**, not as a hard
accept/reject cutoff. This matches Lane 2's own description in the lane guide, "logistic
regression, decision tree, random forest... a ranked queue with reason codes", and it matches
what the starter pipeline in this repo already does end to end.


## 2. Target or proxy

**What I would predict (starter version):** `is_declining_label = (trend_direction == "down")`,
a binary 0/1 label, one per content item.

**Where this label comes from:** it is a **proxy**, not a clean future outcome. `trend_direction`
is itself a derived bucket computed inside the same trailing-90-day window as the features,
it's a rule's summary of the recent past, not something that happened *after* a decision point.
The lane guide flags this directly: "notice its weakness: it is a bucket calculated from the
current window, not a future outcome... treat it that way, not as the ideal capstone target."

**Why I'm using it anyway, for now:** it's the only label available in the starter CSV, it lets me
prove the whole loop (data -> baseline -> model -> ranked queue -> validation) end to end this
week, and it's exactly the signal the starter pipeline already validated with client-holdout
splits. I am **not** treating it as the capstone target.

**What a stronger, non-proxy version would look like** (to build once I move to the warehouse's
daily fact table, per the lane guide's Section 5): a label built from a genuine future window,
e.g. `label = 1 if a page's impressions/clicks fall by more than X% over the NEXT 30 days,
measured using only features from the PRIOR 90 days`. That keeps the feature window and the
target window from overlapping, which the current starter label does not guarantee.


## 3. Success metric

**Metric: Precision@50** (also tracking ROC-AUC and average precision as supporting numbers, but
precision@50 is the one I'd defend as "good").

**Why this one, and not plain accuracy:** the real decision this supports is a reviewer working
down a ranked queue with limited capacity -- realistically the top 20-50 items in a review cycle,
per the lane guide's own framing ("Precision@50 if the team can act on 50 candidates"). Accuracy
across all 30,000 rows doesn't match that decision at all, a model could be 90% "accurate" by
being right about the bulk of stable, unremarkable pages while being useless at the top of the
queue, which is the only part anyone actually reads.

**What "good" looks like, concretely, using numbers already committed to this repo's own pipeline
output** (`outputs/model_report.md`): the rule-based baseline scores precision@50 = 0.240 (about
12 of the top 50 recommendations are right); the random forest scores precision@50 = 0.740 (about
37 of 50 right). So my working bar for "good" this week is: **beat the transparent baseline rule's
precision@50 by a clear, non-trivial margin, under client-holdout validation** -- not just beat 0.5,
and not just beat 0 (the base rate here is already 54.2%, so a naive "always predict declining"
guess would score high on plain accuracy without being useful for ranking at all).


## 4. The unit of analysis, as a real dataframe

Loading my lane's slice of the starter data and applying the same eligibility filters the starter
pipeline uses, so the unit of analysis is unambiguous: **one row = one (pseudonymized) content
item, at one point in time (the trailing-90-day snapshot), for one client.**


In [ ]:
import pandas as pd

# Self-contained: clone if not already present in this environment (Colab-safe)
import os
if not os.path.exists("flyrank_assignment01"):
    os.system("git clone -q https://github.com/omairtahir3/flyrank_assignment01.git")

csv_path = "flyrank_assignment01/data/raw/content_refresh_anonymized.csv" \
    if os.path.exists("flyrank_assignment01/data/raw/content_refresh_anonymized.csv") \
    else "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

# Same eligibility filters the starter pipeline applies: real demand, old enough to trend
lane_slice = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
lane_slice = lane_slice.drop_duplicates(subset="content_id")

# Build the proxy target described in Section 2
lane_slice["is_declining_label"] = (lane_slice["trend_direction"] == "down").astype(int)

print(f"Unit of analysis: one row = one content item, one client, one trailing-90-day snapshot")
print(f"Rows in lane slice: {len(lane_slice):,}  |  unique content_id: {lane_slice['content_id'].nunique():,}"
      f"  |  unique clients: {lane_slice['client_id'].nunique()}")
print(f"Target base rate (is_declining_label == 1): {lane_slice['is_declining_label'].mean():.1%}")
print()

cols_to_show = [
    "content_id", "client_id", "impressions_90d", "avg_position", "ctr",
    "engagement_rate", "content_age_days", "trend_direction", "is_declining_label",
]
lane_slice[cols_to_show].head(8)


Unit of analysis: one row = one content item, one client, one trailing-90-day snapshot
Rows in lane slice: 30,000  |  unique content_id: 30,000  |  unique clients: 32
Target base rate (is_declining_label == 1): 54.2%



,content_id,client_id,impressions_90d,avg_position,ctr,engagement_rate,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,5.88,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,0.00,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,0.00,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,1.28,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,0.00,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,0.00,147,down,1
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,0.00,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,3.57,445,stable,0


## 5. Why ML beats a fixed rule here

The starter baseline **is** the fixed rule: a hand-weighted blend of four sub-scores
(`0.40 * visibility_score + 0.30 * freshness_risk_score + 0.25 * position_opportunity_score +
0.05 * depth_gap_score`) plus a handful of threshold-based reason codes like
`stale_visible_page` (update age >= 180 days AND impressions >= 500) or `declining_with_demand`
(trend down AND impressions >= 100). It's transparent and explainable, but it's also just a
guess at how much each signal should count, fixed in advance, the same for every content type and
client.

The evidence that this guess is measurably wrong: on this exact starter slice, the rule reaches
precision@50 = 0.240 while a random forest, which can learn nonlinear interactions and different
weightings per region of the feature space, reaches precision@50 = 0.740 (both numbers from
`outputs/model_report.md`, already committed in this repo). That's roughly a 3x improvement in
"correct picks in the top 50", not a marginal gain.

Why the gap exists, in plain terms: the signals that matter (position, CTR, freshness, word
count, engagement, trend) don't move independently or linearly. A stale page with low impressions
might not be worth reviewing at all, but a stale page with *high* impressions is a different
animal entirely, and the "high enough to matter" threshold likely differs by content type,
position tier, and client. A single linear weighting can't represent "the effect of freshness
depends on how much traffic is already at stake," but a tree-based model can split on exactly
that kind of interaction. That's precisely the situation the framing-ml-problems skill describes
as ML earning its place: "the pattern is real but too messy to write by hand, many signals,
tangled, shifting over time."
